# Checkpoint 2: duplicates and corrected readings

A reading can arrive twice, or arrive again with a corrected value. Before calculating anything, we need to decide which version was available at the prediction time.

Choose the project `.venv` kernel and run from the top. This notebook loads its own inputs. The [learning journal](../MY_LEARNINGS.md) records the examples and decisions.

In [1]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
import json

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data/events.jsonl").is_file()
             and (p / "src/dispatch_risk/contracts.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the candidate repository.")

def utc(text):
    value = datetime.fromisoformat(text.replace("Z", "+00:00"))
    if value.tzinfo is None:
        raise ValueError("An explicit timezone is required.")
    return value.astimezone(timezone.utc)

with (ROOT / "data/events.jsonl").open() as handle:
    events = [json.loads(line) for line in handle if line.strip()]
versions = {}
for event in events:
    versions.setdefault(event["event_id"], set()).add(event["revision"])
print("Loaded", len(events), "deliveries; no previous notebook is required.")


Loaded 11019 deliveries; no previous notebook is required.


## 1. Which version did we know?

Before calculating features, we must decide which records represent the information available at a checkpoint.

**Duplicate:** the same event and revision delivered again. It should not count as another measurement.

**Correction:** a higher revision of the same event. It replaces the older revision only for checkpoints at or after its received time.

### A small teaching example (invented, not the supplied data)
A reading measures 8°C at 9 AM and arrives at 9:05. Its exact duplicate arrives in the stream again. A correction changes the value to 5°C, but is first available at noon.

| Prediction time | Value to use | Reason |
|---|---|---|
| 9 AM | No reading yet | Original is not available until 9:05 |
| 10 AM | 8°C | Only revision 1 was available |
| Noon | 5°C | Revision 2 is now available |

Using 5°C for the 10 AM decision would leak future knowledge. Keeping both 8°C and 5°C as independent readings would count a single measurement twice.

Our notebook policy: filter by received time, collapse exact duplicates, then select the highest available revision per event ID. Stable sorting of the selected output makes inspection repeatable; it does not reorder online delivery. This helper only resolves historical revisions. Clock validity, feature windows, retention, and complete input validation are separate steps.


In [2]:
def known_revisions(records, checkpoint):
    """Select the highest revision received by the checkpoint for each event ID."""
    if checkpoint.tzinfo is None:
        raise ValueError("Checkpoint must include a timezone")
    checkpoint = checkpoint.astimezone(timezone.utc)
    delivered = {}
    latest = {}
    for event in records:
        if utc(event["received_at"]) > checkpoint:
            continue
        key = (event["event_id"], event["revision"])
        # Compare canonical timestamps so equivalent timezone notation agrees.
        normalized = dict(event)
        for field in ("device_time", "received_at"):
            normalized[field] = utc(event[field]).isoformat()
        signature = json.dumps(normalized, sort_keys=True, allow_nan=False)
        if key in delivered:
            if delivered[key] != signature:
                raise ValueError("Conflicting content for the same event/revision")
            continue
        delivered[key] = signature
        prior = latest.get(event["event_id"])
        if prior is not None and prior["shipment_id"] != event["shipment_id"]:
            raise ValueError("An event ID changed shipment")
        if prior is None or event["revision"] > prior["revision"]:
            latest[event["event_id"]] = dict(event)
    return [latest[event_id] for event_id in sorted(latest)]

original = {
    "event_id": "teaching-reading", "revision": 1,
    "shipment_id": "teaching-shipment",
    "device_time": "2026-01-01T09:00:00Z",
    "received_at": "2026-01-01T09:05:00Z",
    "kind": "temperature_c", "value": 8.0, "source": "teaching-sensor", "payload": {}
}
correction = {**original, "revision": 2, "received_at": "2026-01-01T12:00:00Z", "value": 5.0}
teaching_stream = [original, dict(original), correction]
for hour in (9, 10, 12):
    checkpoint = utc(f"2026-01-01T{hour:02}:00:00Z")
    chosen = known_revisions(teaching_stream, checkpoint)
    print(f"At {hour:02}:00:", [(e["revision"], e["value"]) for e in chosen])

before = utc("2026-01-01T10:00:00Z")
after = utc("2026-01-01T12:00:00Z")
assert known_revisions(teaching_stream, before) == [original]
assert known_revisions(teaching_stream, after) == [correction]
assert known_revisions(teaching_stream * 2, after) == [correction]
assert known_revisions(list(reversed(teaching_stream)), before) == [original]
assert known_revisions(list(reversed(teaching_stream)), after) == [correction]
assert known_revisions([original, correction], before) == known_revisions([original], before)
try:
    known_revisions([original, {**original, "value": 99.0}], before)
except ValueError:
    pass
else:
    raise AssertionError("Conflicting duplicate must not be silently accepted")
print("Passed: duplicates, revisions, boundary, historical order independence, and conflict checks.")


At 09:00: []
At 10:00: [(1, 8.0)]
At 12:00: [(2, 5.0)]
Passed: duplicates, revisions, boundary, historical order independence, and conflict checks.


### Apply it to a correction from the supplied data

Choose an event with multiple revisions from the data rather than depending on a fixed sample ID. Inspect immediately before the higher revision arrived and exactly when it arrived. These are demonstration checkpoints, not necessarily supplied training checkpoints.


In [3]:
revised_ids = sorted(event_id for event_id, revisions in versions.items() if len(revisions) > 1)
if not revised_ids:
    print("This dataset has no multi-revision events; the teaching example still tests the policy.")
else:
    example_id = revised_ids[0]
    revision_records = [e for e in events if e["event_id"] == example_id]
    highest = max(revision_records, key=lambda e: e["revision"])
    arrival = utc(highest["received_at"])
    print("Data example:", example_id)
    for checkpoint in (arrival - timedelta(microseconds=1), arrival):
        selected = known_revisions(revision_records, checkpoint)
        print(checkpoint.isoformat(), "->", [(e["revision"], e["value"]) for e in selected])
    assert known_revisions(revision_records, arrival)[0]["revision"] == highest["revision"]


Data example: s-00000-temp-08
2026-01-01T18:59:59.999999+00:00 -> [(1, 3.908)]
2026-01-01T19:00:00+00:00 -> [(2, 1.658)]


### Why this choice? Explain the decision

“I select the highest revision available at each decision time. A correction received later cannot change the information used for an earlier prediction. I collapse exact duplicates so retries do not add measurements.”

**Alternative rejected:** always using the final corrected dataset, because it reconstructs final truth rather than historical knowledge.

**Provisional conflict policy:** raise an explicit error for contradictory records with the same identity instead of allowing arbitrary file order to decide the value. The final rejection and degraded-scoring rules are documented in DECISIONS.md.

**Limits:** this helper retains its input history in memory and is not the bounded online engine. It has no bad-clock or negative-label policy. Historical dataset reconstruction can ignore file permutation for selecting known revisions, but online ingestion must respect deliveries actually processed.

**Try explaining this:** if the original 8°C reading arrived at 9:05 and its 5°C correction arrived at noon, what should a reconstructed 11 AM prediction use, and why?

**Next checkpoint:** choose usable measurement times and calculate our first temperature/freshness features from the selected readings.
